In [1]:
import torch
import torch.nn as nn
from torchvision import models
from torch.hub import load_state_dict_from_url
from torchvision.models import resnet18, ResNet18_Weights

from PIL import Image
import cv2
import numpy as np
from matplotlib import pyplot as plt

from torchvision import transforms
from torchsummary import summary

In [2]:
class FullyConvolutionalResnet18(models.ResNet):
    def __init__(self, num_classes=1000, pretrained=False, **kwargs):

        # Start with standard resnet18 defined here 
        # https://github.com/pytorch/vision/blob/b2e95657cd5f389e3973212ba7ddbdcc751a7878/torchvision/models/resnet.py
        super().__init__(block=models.resnet.BasicBlock, layers=[2, 2, 2, 2], num_classes=num_classes, **kwargs)
        if pretrained:
            state_dict = torch.load("resnet18.pth", map_location="cpu")
            self.load_state_dict(state_dict)

        # Replace AdaptiveAvgPool2d with standard AvgPool2d 
        # https://github.com/pytorch/vision/blob/b2e95657cd5f389e3973212ba7ddbdcc751a7878/torchvision/models/resnet.py#L153-L154
        self.avgpool = nn.AvgPool2d((7, 7))

        # Add final Convolution Layer. 
        self.last_conv = torch.nn.Conv2d(in_channels=self.fc.in_features, out_channels=num_classes, kernel_size=1)
        self.last_conv.weight.data.copy_(self.fc.weight.data.view(*self.fc.weight.data.shape, 1, 1))
        self.last_conv.bias.data.copy_(self.fc.bias.data)

    # Reimplementing forward pass. 
    # Replacing the following code 
    # https://github.com/pytorch/vision/blob/b2e95657cd5f389e3973212ba7ddbdcc751a7878/torchvision/models/resnet.py#L197-L213
    def _forward_impl(self, x):
        # Standard forward for resnet18
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)

        # Notice, there is no forward pass 
        # through the original fully connected layer. 
        # Instead, we forward pass through the last conv layer
        x = self.last_conv(x)
        return x

In [9]:
# Read ImageNet class id to name mapping
with open('./imagenet_classes.txt') as f:
    labels = [line.strip() for line in f.readlines()]

# Read image
original_image = cv2.imread('Camel4_800.jpg')

# Convert original image to RGB format
image = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)

# Transform input image 
# 1. Convert to Tensor
# 2. Subtract mean
# 3. Divide by standard deviation

transform = transforms.Compose([            
                transforms.ToTensor(), #Convert image to tensor. 
                transforms.Normalize(                      
                mean=[0.485, 0.456, 0.406],   # Subtract mean 
                std=[0.229, 0.224, 0.225]     # Divide by standard deviation             
                )])

image = transform(image)
image = image.unsqueeze(0)
print(image.shape)

# Load modified resnet18 model with pretrained ImageNet weights
model = FullyConvolutionalResnet18(pretrained=True).eval()

torch.Size([1, 3, 533, 800])


In [10]:
from torchinfo import summary
summary(model, input_size=[1, 3, 264, 264]) 

Layer (type:depth-idx)                   Output Shape              Param #
FullyConvolutionalResnet18               [1, 1000, 1, 1]           513,000
├─Conv2d: 1-1                            [1, 64, 132, 132]         9,408
├─BatchNorm2d: 1-2                       [1, 64, 132, 132]         128
├─ReLU: 1-3                              [1, 64, 132, 132]         --
├─MaxPool2d: 1-4                         [1, 64, 66, 66]           --
├─Sequential: 1-5                        [1, 64, 66, 66]           --
│    └─BasicBlock: 2-1                   [1, 64, 66, 66]           --
│    │    └─Conv2d: 3-1                  [1, 64, 66, 66]           36,864
│    │    └─BatchNorm2d: 3-2             [1, 64, 66, 66]           128
│    │    └─ReLU: 3-3                    [1, 64, 66, 66]           --
│    │    └─Conv2d: 3-4                  [1, 64, 66, 66]           36,864
│    │    └─BatchNorm2d: 3-5             [1, 64, 66, 66]           128
│    │    └─ReLU: 3-6                    [1, 64, 66, 66]          

In [4]:
with torch.no_grad():
    # Perform inference.
    # Instead of a 1x1000 vector, we will get a
    # 1x1000xnxm output ( i.e. a probabibility map
    # of size n x m for each 1000 class,
    # where n and m depend on the size of the image.)
    preds = model(image)
    preds = torch.softmax(preds, dim=1)
     
    print('Response map shape : ', preds.shape)
 
    # Find the class with the maximum score in the n x m output map
    pred, class_idx = torch.max(preds, dim=1)
    print(class_idx)
 
    row_max, row_idx = torch.max(pred, dim=1)
    col_max, col_idx = torch.max(row_max, dim=1)
    predicted_class = class_idx[0, row_idx[0, col_idx], col_idx]
     
    # Print top predicted class
    print('Predicted Class : ', labels[predicted_class], predicted_class)

Response map shape :  torch.Size([1, 1000, 2, 3])
tensor([[[978, 970, 672],
         [354, 354, 354]]])
Predicted Class :  Arabian camel, dromedary, Camelus dromedarius tensor([354])


In [5]:
# Find the n x m score map for the predicted class
score_map = preds[0, predicted_class, :, :].cpu().numpy()
score_map = score_map[0]
 
# Resize score map to the original image size
score_map = cv2.resize(score_map, (original_image.shape[1], original_image.shape[0]))
 
# Binarize score map
_, score_map_for_contours = cv2.threshold(score_map, 0.25, 1, type=cv2.THRESH_BINARY)
score_map_for_contours = score_map_for_contours.astype(np.uint8).copy()
 
# Find the countour of the binary blob
contours, _ = cv2.findContours(score_map_for_contours, mode=cv2.RETR_EXTERNAL, method=cv2.CHAIN_APPROX_SIMPLE)
 
# Find bounding box around the object.
rect = cv2.boundingRect(contours[0])

In [6]:
# Apply score map as a mask to original image
score_map = score_map - np.min(score_map[:])
score_map = score_map / np.max(score_map[:])

score_map = cv2.cvtColor(score_map, cv2.COLOR_GRAY2BGR)
masked_image = (original_image * score_map).astype(np.uint8)
 
# Display bounding box
cv2.rectangle(masked_image, rect[:2], (rect[0] + rect[2], rect[1] + rect[3]), (0, 0, 255), 2)
 
# Display images
cv2.imshow("Original Image", original_image)
cv2.imshow("scaled_score_map", score_map)
cv2.imshow("activations_and_bbox", masked_image)
cv2.waitKey(0)

QFontDatabase: Cannot find font directory /home/beni/miniconda3/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/beni/miniconda3/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/beni/miniconda3/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/beni/miniconda3/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/beni/miniconda3/lib/python3.

-1

In [7]:
!nvidia-smi

Thu Apr 23 23:20:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Quadro T2000                   Off |   00000000:01:00.0 Off |                  N/A |
| N/A   52C    P8              2W /   30W |     219MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----